# Survey-Informed Ride-Pooling Operational Model
## Final Google Colab — Unified Model + Scenarios A/B/C

This notebook implements the revised mathematical model and the revised pricing–acceptance interface.

### Core operational sequence

\[
\text{Request}
\rightarrow
\text{Passenger profile}
\rightarrow
\text{Route-feasible candidates}
\rightarrow
\text{Upfront pooled offer}
\rightarrow
\text{Passenger acceptance}
\rightarrow
\text{Minimum-}\Delta D\text{ acceptable insertion}
\rightarrow
\text{Realised route}
\rightarrow
\gamma\text{-based settlement}.
\]

The three scenarios use **one identical routing / insertion / pricing model**.

- **Scenario A — Universal acceptance**
  \[
  \delta_i=0,\qquad WTP_i^{trip}=+\infty
  \]

- **Scenario B — Homogeneous behaviour**
  \[
  \delta_i=\bar{\delta},\qquad
  WTP_i^{trip}=\bar{\omega}P_i^{solo}
  \]

- **Scenario C — Survey-informed heterogeneous behaviour**
  \[
  \delta_i=\delta_i^{survey},\qquad
  WTP_i^{trip}=\omega_i^{survey}P_i^{solo}
  \]

Only the passenger behavioural parameterisation changes.

### Revised pricing logic

For every route-feasible candidate, the platform first gives an **upfront pooled-service offer**

\[
P_{ic}^{offer}=(1-\rho)P_i^{solo},
\]

with offered discount

\[
d_{ic}^{offer}=\rho.
\]

Passenger acceptance is determined **before final route commitment**:

\[
d_{ic}^{offer}\ge\delta_i,
\qquad
P_{ic}^{offer}\le WTP_i^{trip}.
\]

After the realised route is executed, the original undergraduate \(\gamma\)-based sharing-price logic is retained as a settlement rule:

\[
P_i^{final}=\min(P_i^{offer},P_i^\gamma).
\]

This removes the previous bootstrap problem in which the first passenger could not receive a pooling discount before another passenger was already sharing the vehicle.

## 0. Google Colab setup

Run this cell once. `pyreadstat` is used to read the encoded SPSS `.sav` survey directly.

In [ ]:
!pip -q install pyreadstat

## 1. Imports and global settings

The original undergraduate case-study assumptions are retained where possible:

- Euclidean synthetic spatial network;
- travel time = 1.25 × physical distance;
- Beijing-style simplified solo fare;
- original `sigma = 0.5` service-time slack when separate wait/detour limits are not supplied.

The new paper can later replace these case-study assumptions without changing the A/B/C behavioural architecture.

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TIME_FACTOR = 1.25

BASE_FARE = 13.0
BASE_DISTANCE_KM = 3.0
PER_KM_FARE = 2.3

SURVEY_SOLO_FARE_RMB = 30.0

# Q34 option "20 RMB or above" has no natural upper bound.
# The open-ended upper category requires an explicit sensitivity-tested cap.
Q34_OPEN_UPPER_CAP_RMB = 30.0

USE_LEGACY_TIME_SLACK_WHEN_LIMITS_MISSING = True

# Example treatment for interval-valued Q34:
# sample a value inside each reported interval.
MAIN_WTP_METHOD = "random"

# Reproducibility
BASE_SEED = 2026

print("Configuration loaded.")

## 2. Core data structures and operational engine

This section contains the tested unified insertion algorithm.

Important corrections relative to the old undergraduate code are retained:

1. physical distance and travel time are separated;
2. vehicle state advances with request time;
3. capacity is checked using actual cumulative onboard passengers;
4. route candidates are generated **without committing the vehicle route**;
5. route commitment occurs only after passenger acceptance;
6. realised shared/non-shared distance is accumulated during actual vehicle movement;
7. the final \(\gamma\)-based fare is calculated only after the route is realised.

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any
import math
import numpy as np
import pandas as pd

TIME_FACTOR=1.25
BASE_FARE=13.0
BASE_DISTANCE_KM=3.0
PER_KM_FARE=2.3
USE_LEGACY_TIME_SLACK_WHEN_LIMITS_MISSING=True
SURVEY_SOLO_FARE_RMB=30.0

@dataclass(frozen=True)
class Location:
    x: float; y: float

@dataclass
class PassengerProfile:
    respondent_id: Any=None
    delta: float=0.0
    omega: float=math.inf
    never_accept: bool=False
    share_willingness_score: Optional[int]=None
    q16_code: Optional[int]=None
    q29_code: Optional[int]=None
    q33_code: Optional[int]=None
    q34_code: Optional[int]=None
    wtp_survey_rmb: Optional[float]=None

@dataclass
class Request:
    request_id:int; t_r:float; o_r:Location; d_r:Location; party_size:int
    sigma:float=0.5
    max_wait:Optional[float]=None
    max_detour:Optional[float]=None
    profile:Optional[PassengerProfile]=None
    accepted:bool=False
    nominal_feasible:bool=False
    rejection_reason:Optional[str]=None
    assigned_car_id:Optional[int]=None
    offered_fare:Optional[float]=None
    offered_discount:Optional[float]=None
    delta_vkt:Optional[float]=None
    pickup_time_actual:Optional[float]=None
    dropoff_time_actual:Optional[float]=None
    actual_distance:float=0.0
    shared_distance:float=0.0
    nonshared_distance:float=0.0
    gamma:Optional[float]=None
    gamma_fare:Optional[float]=None
    final_fare:Optional[float]=None
    @property
    def direct_distance(self): return physical_distance(self.o_r,self.d_r)
    @property
    def direct_travel_time(self): return travel_time(self.o_r,self.d_r)
    @property
    def solo_fare(self): return solo_fare_from_distance(self.direct_distance)

@dataclass
class CarNode:
    location:Location; kind:str; request_id:int

@dataclass
class Car:
    car_id:int; current_location:Location; capacity:int; current_time:float=0.0
    route:List[CarNode]=field(default_factory=list)
    onboard:set=field(default_factory=set)
    pickup_times:Dict[int,float]=field(default_factory=dict)
    assigned_request_ids:set=field(default_factory=set)
    distance_travelled:float=0.0; occupied_vkt:float=0.0; passenger_km:float=0.0

@dataclass
class Candidate:
    car_id:int; route:List[CarNode]; delta_vkt:float; offer_fare:float; offer_discount:float; acceptable:bool

def physical_distance(a,b): return math.hypot(a.x-b.x,a.y-b.y)
def travel_time(a,b): return TIME_FACTOR*physical_distance(a,b)
def solo_fare_from_distance(d):
    if d<=0:return 0.0
    if d<=BASE_DISTANCE_KM:return BASE_FARE
    return BASE_FARE+(d-BASE_DISTANCE_KM)*PER_KM_FARE

def route_distance_from_state(car,route):
    total=0; cur=car.current_location
    for node in route:
        total+=physical_distance(cur,node.location);cur=node.location
    return total

def upfront_offer(request,rho):
    rho=float(rho)
    if not (0<=rho<=1): raise ValueError
    return (1-rho)*request.solo_fare, rho

def candidate_accepts(request, offer_fare, offer_discount, mode, homogeneous_params=None):
    if mode=='universal': return True
    if mode=='homogeneous':
        d=homogeneous_params['delta']; o=homogeneous_params['omega']
        return offer_discount+1e-12>=d and offer_fare<=o*request.solo_fare+1e-12
    if mode=='survey_heterogeneous':
        p=request.profile
        if p is None: raise ValueError
        if p.never_accept:return False
        return offer_discount+1e-12>=p.delta and offer_fare<=p.omega*request.solo_fare+1e-12
    raise ValueError

def simulate_route_schedule(car,route,requests):
    ct=car.current_time; cl=car.current_location
    active=set(car.onboard); pickup_times=dict(car.pickup_times); dropoff_times={}
    occ=sum(requests[r].party_size for r in active)
    if occ>car.capacity:return None
    for node in route:
        ct+=travel_time(cl,node.location);cl=node.location;req=requests[node.request_id]
        if node.kind=='pickup':
            if ct<req.t_r:ct=req.t_r
            if node.request_id in active:return None
            active.add(node.request_id);pickup_times[node.request_id]=ct
            occ=sum(requests[r].party_size for r in active)
            if occ>car.capacity:return None
        elif node.kind=='dropoff':
            if node.request_id not in active:return None
            dropoff_times[node.request_id]=ct;active.remove(node.request_id)
    for rid,dt in dropoff_times.items():
        req=requests[rid];pt=pickup_times.get(rid)
        if pt is None:return None
        wait=pt-req.t_r; pooled=dt-pt
        if req.max_wait is not None and wait>req.max_wait+1e-9:return None
        if req.max_detour is not None and pooled-req.direct_travel_time>req.max_detour+1e-9:return None
        if USE_LEGACY_TIME_SLACK_WHEN_LIMITS_MISSING and req.max_wait is None and req.max_detour is None:
            if dt-req.t_r>(1+req.sigma)*req.direct_travel_time+1e-9:return None
    return {'pickup_times':pickup_times,'dropoff_times':dropoff_times,'end_time':ct}

def generate_candidates_for_car(car,request,requests,mode,rho,hom=None):
    base=list(car.route);out=[];n=len(base)
    of,od=upfront_offer(request,rho)
    acc=candidate_accepts(request,of,od,mode,hom)
    for pp in range(n+1):
        rp=base[:pp]+[CarNode(request.o_r,'pickup',request.request_id)]+base[pp:]
        for dp in range(pp+1,len(rp)+1):
            cand=rp[:dp]+[CarNode(request.d_r,'dropoff',request.request_id)]+rp[dp:]
            if simulate_route_schedule(car,cand,requests) is None: continue
            dv=route_distance_from_state(car,cand)-route_distance_from_state(car,base)
            out.append(Candidate(car.car_id,cand,dv,of,od,acc))
    return out

def _accumulate_segment(car, moved_distance, requests):
    if moved_distance<=0:return
    active=list(car.onboard)
    onboard_people=sum(requests[r].party_size for r in active)
    car.distance_travelled += moved_distance
    car.passenger_km += moved_distance*onboard_people
    if onboard_people>0:car.occupied_vkt += moved_distance
    n_active_requests=len(active)
    for rid in active:
        req=requests[rid]
        req.actual_distance += moved_distance
        if n_active_requests>=2:req.shared_distance += moved_distance
        else:req.nonshared_distance += moved_distance

def advance_car_to_time(car,target,requests):
    eps=1e-9
    if target<car.current_time-eps:raise ValueError
    while car.current_time<target-eps:
        if not car.route:car.current_time=target;return
        node=car.route[0];dist=physical_distance(car.current_location,node.location);tt=travel_time(car.current_location,node.location);arr=car.current_time+tt
        rel=requests[node.request_id].t_r if node.kind=='pickup' else None
        if target<arr-eps:
            frac=(target-car.current_time)/tt if tt>0 else 1.0
            nl=Location(car.current_location.x+frac*(node.location.x-car.current_location.x),car.current_location.y+frac*(node.location.y-car.current_location.y))
            md=physical_distance(car.current_location,nl);_accumulate_segment(car,md,requests)
            car.current_location=nl;car.current_time=target;return
        _accumulate_segment(car,dist,requests);car.current_location=node.location;car.current_time=arr
        if rel is not None and car.current_time<rel-eps:
            if target<rel-eps:car.current_time=target;return
            car.current_time=rel
        if node.kind=='pickup':
            car.onboard.add(node.request_id);car.pickup_times[node.request_id]=car.current_time;requests[node.request_id].pickup_time_actual=car.current_time
        else:
            car.onboard.discard(node.request_id);requests[node.request_id].dropoff_time_actual=car.current_time;car.pickup_times.pop(node.request_id,None)
        car.route.pop(0)
    car.current_time=target

def finish_car_route(car,requests):
    while car.route:
        node=car.route[0];target=car.current_time+travel_time(car.current_location,node.location)
        if node.kind=='pickup':target=max(target,requests[node.request_id].t_r)
        advance_car_to_time(car,target,requests)

def gamma_settlement(req):
    if not req.accepted:return
    lt=req.actual_distance; ls=req.shared_distance; ln=req.nonshared_distance
    if lt<=1e-12:
        pg=0.0;g=0.0
    else:
        g=(ln**2)/(lt**2)
        if lt<=BASE_DISTANCE_KM:
            pg=BASE_FARE
        elif ls<=BASE_DISTANCE_KM:
            pg=BASE_FARE*g+PER_KM_FARE*(lt-BASE_DISTANCE_KM)
        else:
            pg=BASE_FARE+PER_KM_FARE*(lt-BASE_DISTANCE_KM)*g
    req.gamma=g;req.gamma_fare=pg;req.final_fare=min(req.offered_fare,pg)

def run_simulation(cars,request_list,mode='universal',rho=0.75,hom=None):
    requests={r.request_id:r for r in request_list};cd={c.car_id:c for c in cars}
    for req in sorted(request_list,key=lambda r:(r.t_r,r.request_id)):
        for car in cd.values():advance_car_to_time(car,req.t_r,requests)
        feasible=[]; acceptable=[]
        for car in cd.values():
            cs=generate_candidates_for_car(car,req,requests,mode,rho,hom);feasible+=cs;acceptable += [c for c in cs if c.acceptable]
        req.nominal_feasible=bool(feasible)
        if not feasible:req.rejection_reason='route_infeasible';continue
        if not acceptable:req.rejection_reason='passenger_unacceptable';continue
        best=min(acceptable,key=lambda c:c.delta_vkt);car=cd[best.car_id];car.route=best.route;car.assigned_request_ids.add(req.request_id)
        req.accepted=True;req.assigned_car_id=best.car_id;req.offered_fare=best.offer_fare;req.offered_discount=best.offer_discount;req.delta_vkt=best.delta_vkt
    for car in cd.values():finish_car_route(car,requests)
    for r in request_list:gamma_settlement(r)
    n=len(request_list);real=sum(r.accepted for r in request_list);nom=sum(r.nominal_feasible for r in request_list)
    total_vkt=sum(c.distance_travelled for c in cd.values());occ=sum(c.occupied_vkt for c in cd.values());pkm=sum(c.passenger_km for c in cd.values())
    offerrev=sum(r.offered_fare or 0 for r in request_list if r.accepted);finalrev=sum(r.final_fare or 0 for r in request_list if r.accepted)
    return {'summary':{'mode':mode,'rho':rho,'requests':n,'realised_services':real,'nominal_matching_rate':nom/n,'realised_service_rate':real/n,'route_infeasible_rejections':sum(r.rejection_reason=='route_infeasible' for r in request_list),'passenger_unacceptable_rejections':sum(r.rejection_reason=='passenger_unacceptable' for r in request_list),'total_vkt':total_vkt,'upfront_offer_revenue':offerrev,'final_fare_revenue':finalrev,'occupied_vkt_share':occ/total_vkt if total_vkt else np.nan,'distance_weighted_occupancy':pkm/total_vkt if total_vkt else np.nan},'requests':request_list,'cars':cd}

def build_original_case_study():
    cars=[Car(0,Location(6.7,5.6),4),Car(1,Location(66.7,10.1),4),Car(2,Location(11.5,15.5),4),Car(3,Location(2.9,15.2),3),Car(4,Location(15.5,9.3),4),Car(5,Location(50.6,36.4),4)]
    rr=[(0,(5,6.7),(28.3,6.7),1),(5,(11.8,14.3),(18.8,13.5),2),(5,(50.4,37.1),(51.6,33.5),2),(5,(4.8,7.1),(53.5,25.2),1),(5,(50.6,38),(50.2,20),2),(10,(16.9,9.7),(36.1,18.9),2),(10,(6.2,10.4),(59.4,17.3),2),(10,(60.7,9.8),(45.2,16),1),(15,(11.4,16.5),(32.3,19.8),1),(15,(13.9,7.5),(50.2,17.4),1),(15,(30.9,5.3),(42.3,23.2),1),(15,(9,10.9),(35.5,20.7),1),(15,(49.5,36.3),(32.8,32.9),2),(15,(3.5,5.5),(30.6,16.4),1),(20,(48.9,38),(28.3,13.3),2),(20,(4.8,10),(65.8,11.1),2),(20,(4.5,6.3),(31.2,36.5),1),(20,(43.3,20.2),(50.2,17.4),2),(20,(8.9,12.3),(37.2,20.9),1),(40,(62.3,16.7),(26.3,28.8),1)]
    req=[Request(i,t,Location(*o),Location(*d),k) for i,(t,o,d,k) in enumerate(rr)]
    return cars,req

## 3. Algorithm corresponding to the revised mathematical model

For each incoming request \(i\):

1. advance every vehicle to \(t_i^r\);
2. enumerate every feasible pickup/drop-off insertion for every vehicle;
3. check operational feasibility \(F_{ic}\);
4. calculate \(\Delta D_{ic}\);
5. generate the common upfront offer \(P_{ic}^{offer}\);
6. apply Scenario A/B/C passenger acceptance;
7. construct
   \[
   C_i^A=\{c:F_{ic}=1,\;P_{ic}^{offer}\le WTP_i^{trip},\;
   d_{ic}^{offer}\ge\delta_i\};
   \]
8. if \(C_i^A\neq\varnothing\), choose
   \[
   c_i^*=\arg\min_{c\in C_i^A}\Delta D_{ic};
   \]
9. otherwise classify the rejection as route-infeasible or passenger-unacceptable;
10. after all requests, execute remaining routes and calculate
    \[
    P_i^{final}=\min(P_i^{offer},P_i^\gamma).
    \]

The code above implements this sequence directly.

## 4. Original undergraduate 6-vehicle / 20-request case

The active vehicle and request values are copied from the uploaded original Python files.

This small case is used only as an initial validation case. It should not be treated as the final paper experiment.

In [ ]:
cars_check, requests_check = build_original_case_study()

print("Vehicles:", len(cars_check))
print("Requests:", len(requests_check))

case_df = pd.DataFrame([
    {
        "request_id": r.request_id,
        "request_time": r.t_r,
        "origin": (r.o_r.x, r.o_r.y),
        "destination": (r.d_r.x, r.d_r.y),
        "party_size": r.party_size,
        "direct_distance": r.direct_distance,
        "solo_fare": r.solo_fare,
    }
    for r in requests_check
])

display(case_df)

## 5. Upload an authorised encoded SPSS survey

Upload:

`survey_encoded.sav`

The model uses the raw SPSS numerical codes. No manual CSV conversion is required.

In [ ]:
from google.colab import files
import pyreadstat

uploaded = files.upload()

sav_files = [
    filename
    for filename in uploaded.keys()
    if filename.lower().endswith(".sav")
]

if not sav_files:
    raise ValueError("Please upload the encoded .sav survey file.")

sav_path = sav_files[0]

survey_raw, survey_meta = pyreadstat.read_sav(
    sav_path,
    apply_value_formats=False
)

required_cols = ["Q16", "Q29", "Q33", "Q34"]

missing = [
    col for col in required_cols
    if col not in survey_raw.columns
]

if missing:
    raise ValueError(f"Missing required survey variables: {missing}")

survey = survey_raw[required_cols].copy()

print("Survey file:", sav_path)
print("Respondents:", len(survey))
display(survey.head())

## 6. Validate the supplied questionnaire coding

The mapping follows the questionnaire appendix.

### Q16 — minimum acceptable pooling discount

Question options:

- code 1: fare ratio 0.75 → minimum discount \(25\%\);
- code 2: fare ratio 0.65 → minimum discount \(35\%\);
- code 3: fare ratio 0.55 → minimum discount \(45\%\);
- code 4: never consider pooling regardless of discount.

### Q29 — willingness to share with strangers

- code 1 = very willing;
- code 2 = willing;
- code 3 = neutral;
- code 4 = unwilling;
- code 5 = very unwilling.

For descriptive consistency we store

\[
S_i=6-Q29_i,
\]

so larger \(S_i\) means greater willingness.  
Q29 is **not added as another hard acceptance constraint** in the main operational model.

### Q33–Q34 — WTP

Q33:
- code 1 = willing to pay;
- code 2 = unwilling to pay.

Q34:
- code 1 = below 5 RMB;
- code 2 = 5–10 RMB;
- code 3 = 10–20 RMB;
- code 4 = above 20 RMB.

If Q33 = 2, the main specification sets stated WTP to zero.

For Q34 interval observations:

\[
\omega_i=\frac{WTP_i^{survey}}{30},
\qquad
WTP_i^{trip}=\omega_iP_i^{solo}.
\]

In [ ]:
print("Q16 distribution:")
print(survey["Q16"].value_counts(dropna=False).sort_index())

print("\nQ29 distribution:")
print(survey["Q29"].value_counts(dropna=False).sort_index())

print("\nQ33 distribution:")
print(survey["Q33"].value_counts(dropna=False).sort_index())

print("\nQ34 distribution:")
print(survey["Q34"].value_counts(dropna=False).sort_index())



## 7. Convert the survey to respondent-level behavioural profiles

Scenario C samples **one complete respondent row with replacement** for each simulated request.  
Q16/Q29/Q33/Q34 are therefore never sampled independently.

In [ ]:
def build_survey_profiles(
    survey_df,
    wtp_method=MAIN_WTP_METHOD,
    seed=BASE_SEED,
    upper_cap=Q34_OPEN_UPPER_CAP_RMB,
):
    rng = np.random.default_rng(seed)
    profiles = []

    q16_delta = {
        1: 0.25,
        2: 0.35,
        3: 0.45,
        4: 0.00,
    }

    q34_intervals = {
        1: (0.0, 5.0),
        2: (5.0, 10.0),
        3: (10.0, 20.0),
        4: (20.0, float(upper_cap)),
    }

    for respondent_id, row in survey_df.iterrows():

        q16 = int(row["Q16"])
        q29 = int(row["Q29"])
        q33 = int(row["Q33"])

        never_accept = (q16 == 4)
        delta = q16_delta[q16]

        if q33 == 2 or pd.isna(row["Q34"]):
            q34 = 0
            wtp_survey = 0.0

        else:
            q34 = int(row["Q34"])
            lower, upper = q34_intervals[q34]

            if wtp_method == "random":
                wtp_survey = float(
                    rng.uniform(lower, upper)
                )

            elif wtp_method == "midpoint":
                wtp_survey = (lower + upper) / 2

            elif wtp_method == "lower":
                wtp_survey = lower

            elif wtp_method == "upper":
                wtp_survey = upper

            else:
                raise ValueError(
                    "wtp_method must be random/midpoint/lower/upper"
                )

        omega = wtp_survey / SURVEY_SOLO_FARE_RMB

        profiles.append(
            PassengerProfile(
                respondent_id=respondent_id,
                delta=delta,
                omega=omega,
                never_accept=never_accept,
                share_willingness_score=6 - q29,
                q16_code=q16,
                q29_code=q29,
                q33_code=q33,
                q34_code=q34,
                wtp_survey_rmb=wtp_survey,
            )
        )

    return profiles


def assign_complete_profiles(
    request_list,
    profiles,
    seed=BASE_SEED,
):
    rng = np.random.default_rng(seed)

    sampled_indices = rng.integers(
        0,
        len(profiles),
        size=len(request_list)
    )

    for request, profile_index in zip(
        request_list,
        sampled_indices
    ):
        request.profile = profiles[
            profile_index
        ]

    return request_list


profiles_main = build_survey_profiles(
    survey,
    wtp_method=MAIN_WTP_METHOD,
    seed=BASE_SEED,
)

profiles_midpoint = build_survey_profiles(
    survey,
    wtp_method="midpoint",
    seed=BASE_SEED,
)

profile_df = pd.DataFrame([
    {
        "respondent_id": p.respondent_id,
        "Q16": p.q16_code,
        "delta_i": p.delta,
        "never_accept": p.never_accept,
        "Q29": p.q29_code,
        "share_willingness_score": p.share_willingness_score,
        "Q33": p.q33_code,
        "Q34": p.q34_code,
        "WTP_survey_RMB": p.wtp_survey_rmb,
        "omega_i": p.omega,
    }
    for p in profiles_main
])

display(profile_df.head())

## 8. Scenario B — homogeneous parameters derived from the supplied sample

To avoid tuning Scenario B after seeing operational results:

- \(ar{\delta}\) = median finite Q16 threshold;
- \(ar{\omega}\) = median respondent WTP ratio using the deterministic Q34 midpoint representation.

The `never accept` category does not have a finite discount threshold, so it is excluded only when calculating the representative finite \(ar{\delta}\).

In [ ]:
finite_delta = np.array([
    p.delta
    for p in profiles_midpoint
    if not p.never_accept
])

omega_midpoint = np.array([
    p.omega
    for p in profiles_midpoint
])

homogeneous_params = {
    "delta": float(np.median(finite_delta)),
    "omega": float(np.median(omega_midpoint)),
}

print("Scenario B homogeneous parameters:")
print(homogeneous_params)

print(
    "Equivalent median survey WTP =",
    homogeneous_params["omega"] * SURVEY_SOLO_FARE_RMB,
    "RMB"
)

# Mechanically derived break-even upfront discount for Scenario B:
# rho must satisfy rho >= delta_bar and 1-rho <= omega_bar.
ILLUSTRATIVE_RHO = max(
    homogeneous_params["delta"],
    1 - homogeneous_params["omega"],
)

print(
    "Homogeneous break-even upfront discount rho =",
    round(ILLUSTRATIVE_RHO, 4)
)

## 9. Run Scenario A and Scenario B at the illustrative common pricing environment

The notebook does **not** choose the illustrative discount to improve the result.

It is derived mechanically from the homogeneous survey parameters:

\[
\rho_{\text{break-even}}
=
\max\left(\bar{\delta},1-\bar{\omega}\right).
\]

Its value is computed from the locally supplied survey rather than fixed in the public notebook.

All three scenarios receive exactly the same \(\rho\).

In [ ]:
rho = ILLUSTRATIVE_RHO

cars_A, requests_A = build_original_case_study()
result_A = run_simulation(
    cars_A,
    requests_A,
    mode="universal",
    rho=rho,
    hom=homogeneous_params,
)

cars_B, requests_B = build_original_case_study()
result_B = run_simulation(
    cars_B,
    requests_B,
    mode="homogeneous",
    rho=rho,
    hom=homogeneous_params,
)

display(
    pd.DataFrame(
        [result_A["summary"], result_B["summary"]],
        index=["A Universal", "B Homogeneous"]
    )
)

## 10. Scenario C — one reproducible realisation

This is a single run only, useful for checking request-level logic.  
Formal interpretation should use Monte Carlo results in the next section.

In [ ]:
profiles_C = build_survey_profiles(
    survey,
    wtp_method=MAIN_WTP_METHOD,
    seed=BASE_SEED,
)

cars_C, requests_C = build_original_case_study()

requests_C = assign_complete_profiles(
    requests_C,
    profiles_C,
    seed=BASE_SEED,
)

result_C_single = run_simulation(
    cars_C,
    requests_C,
    mode="survey_heterogeneous",
    rho=rho,
    hom=homogeneous_params,
)

single_summary = pd.DataFrame(
    [
        result_A["summary"],
        result_B["summary"],
        result_C_single["summary"],
    ],
    index=[
        "A Universal",
        "B Homogeneous",
        "C Survey-informed (single seed)"
    ]
)

display(single_summary)

In [ ]:
request_level_C = pd.DataFrame([
    {
        "request_id": r.request_id,
        "respondent_id": (
            r.profile.respondent_id
            if r.profile else None
        ),
        "delta_i": (
            r.profile.delta
            if r.profile else None
        ),
        "omega_i": (
            r.profile.omega
            if r.profile else None
        ),
        "WTP_survey_RMB": (
            r.profile.wtp_survey_rmb
            if r.profile else None
        ),
        "never_accept": (
            r.profile.never_accept
            if r.profile else None
        ),
        "nominal_feasible": r.nominal_feasible,
        "accepted": r.accepted,
        "rejection_reason": r.rejection_reason,
        "car_id": r.assigned_car_id,
        "solo_fare": r.solo_fare,
        "offered_discount": r.offered_discount,
        "offered_fare": r.offered_fare,
        "actual_distance": r.actual_distance,
        "shared_distance": r.shared_distance,
        "gamma": r.gamma,
        "gamma_fare": r.gamma_fare,
        "final_fare": r.final_fare,
    }
    for r in requests_C
])

display(request_level_C)

## 11. Scenario C Monte Carlo

Because the original case has only 20 requests, a single profile assignment is noisy.

This cell repeats Scenario C while changing:

1. respondent-profile sampling;
2. within-interval Q34 WTP draws.

The operational demand, vehicles, routing, constraints and pricing policy remain unchanged.

In [ ]:
def monte_carlo_scenario_c(
    survey_df,
    rho,
    replications=300,
    base_seed=10000,
    wtp_method=MAIN_WTP_METHOD,
):
    rows = []

    for rep in range(replications):

        profiles = build_survey_profiles(
            survey_df,
            wtp_method=wtp_method,
            seed=base_seed + rep,
        )

        cars, requests = build_original_case_study()

        requests = assign_complete_profiles(
            requests,
            profiles,
            seed=base_seed + 100000 + rep,
        )

        result = run_simulation(
            cars,
            requests,
            mode="survey_heterogeneous",
            rho=rho,
            hom=homogeneous_params,
        )

        row = dict(result["summary"])
        row["replication"] = rep
        rows.append(row)

    return pd.DataFrame(rows)


mc_C = monte_carlo_scenario_c(
    survey,
    rho=rho,
    replications=300,
    base_seed=10000,
)

mc_summary = pd.DataFrame({
    "mean": mc_C.mean(numeric_only=True),
    "std": mc_C.std(numeric_only=True),
    "p2.5": mc_C.quantile(0.025, numeric_only=True),
    "p97.5": mc_C.quantile(0.975, numeric_only=True),
})

display(
    mc_summary.loc[
        [
            "nominal_matching_rate",
            "realised_service_rate",
            "route_infeasible_rejections",
            "passenger_unacceptable_rejections",
            "total_vkt",
            "upfront_offer_revenue",
            "final_fare_revenue",
            "occupied_vkt_share",
            "distance_weighted_occupancy",
        ]
    ]
)

## 12. A/B/C summary at the illustrative \(
ho\)

Scenario A and B are deterministic in this 20-request case.  
Scenario C is reported using its Monte Carlo mean and empirical 95% interval.

In [ ]:
C_mean = mc_C.mean(numeric_only=True)

ABC_summary = pd.DataFrame([
    {
        "scenario": "A Universal",
        **result_A["summary"],
    },
    {
        "scenario": "B Homogeneous",
        **result_B["summary"],
    },
    {
        "scenario": "C Survey-informed (MC mean)",
        **{
            key: C_mean[key]
            for key in [
                "rho",
                "requests",
                "nominal_matching_rate",
                "realised_service_rate",
                "route_infeasible_rejections",
                "passenger_unacceptable_rejections",
                "total_vkt",
                "upfront_offer_revenue",
                "final_fare_revenue",
                "occupied_vkt_share",
                "distance_weighted_occupancy",
            ]
        }
    },
])

display(ABC_summary)

## 13. Upfront-discount sensitivity

This is important because the revised model explicitly separates the pricing environment from passenger behavioural assumptions.

The same discount \(\rho\) is applied to A/B/C at each point.

Suggested initial grid:

\[
\rho\in
\{0.35,0.45,0.55,0.65,0.75,0.85\}.
\]

Scenario C is averaged over repeated respondent assignments.

In [ ]:
def run_discount_sensitivity(
    survey_df,
    rho_grid=(0.35, 0.45, 0.55, 0.65, 0.75, 0.85),
    c_replications=100,
    base_seed=30000,
):
    rows = []

    for rho_value in rho_grid:

        # Scenario A
        cars_Ai, requests_Ai = build_original_case_study()
        result_Ai = run_simulation(
            cars_Ai,
            requests_Ai,
            mode="universal",
            rho=rho_value,
            hom=homogeneous_params,
        )

        rows.append({
            "rho": rho_value,
            "scenario": "A Universal",
            "realised_service_rate":
                result_Ai["summary"]["realised_service_rate"],
            "nominal_matching_rate":
                result_Ai["summary"]["nominal_matching_rate"],
            "total_vkt":
                result_Ai["summary"]["total_vkt"],
            "final_fare_revenue":
                result_Ai["summary"]["final_fare_revenue"],
            "service_rate_low":
                result_Ai["summary"]["realised_service_rate"],
            "service_rate_high":
                result_Ai["summary"]["realised_service_rate"],
        })

        # Scenario B
        cars_Bi, requests_Bi = build_original_case_study()
        result_Bi = run_simulation(
            cars_Bi,
            requests_Bi,
            mode="homogeneous",
            rho=rho_value,
            hom=homogeneous_params,
        )

        rows.append({
            "rho": rho_value,
            "scenario": "B Homogeneous",
            "realised_service_rate":
                result_Bi["summary"]["realised_service_rate"],
            "nominal_matching_rate":
                result_Bi["summary"]["nominal_matching_rate"],
            "total_vkt":
                result_Bi["summary"]["total_vkt"],
            "final_fare_revenue":
                result_Bi["summary"]["final_fare_revenue"],
            "service_rate_low":
                result_Bi["summary"]["realised_service_rate"],
            "service_rate_high":
                result_Bi["summary"]["realised_service_rate"],
        })

        # Scenario C
        mc = monte_carlo_scenario_c(
            survey_df,
            rho=rho_value,
            replications=c_replications,
            base_seed=base_seed + int(rho_value * 1000),
        )

        rows.append({
            "rho": rho_value,
            "scenario": "C Survey-informed",
            "realised_service_rate":
                mc["realised_service_rate"].mean(),
            "nominal_matching_rate":
                mc["nominal_matching_rate"].mean(),
            "total_vkt":
                mc["total_vkt"].mean(),
            "final_fare_revenue":
                mc["final_fare_revenue"].mean(),
            "service_rate_low":
                mc["realised_service_rate"].quantile(0.025),
            "service_rate_high":
                mc["realised_service_rate"].quantile(0.975),
        })

    return pd.DataFrame(rows)


discount_results = run_discount_sensitivity(
    survey,
    c_replications=100,
)

display(discount_results)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for scenario, group in discount_results.groupby("scenario"):
    group = group.sort_values("rho")

    ax.plot(
        group["rho"],
        group["realised_service_rate"],
        marker="o",
        label=scenario,
    )

ax.set_xlabel("Upfront pooled discount rho")
ax.set_ylabel("Realised service rate")
ax.set_title("A/B/C service rate under common pricing environments")
ax.set_ylim(0, 1)
ax.grid(alpha=0.25)
ax.legend()

plt.show()

## 14. Q34 interval sensitivity for Scenario C

The main model samples within the reported interval.  
Lower, midpoint and upper treatments can be checked without changing any routing or pricing rule.

In [ ]:
def run_wtp_interpretation_sensitivity(
    survey_df,
    rho,
    methods=("lower", "midpoint", "upper", "random"),
    replications=100,
):
    rows = []

    for method_index, method in enumerate(methods):

        mc = monte_carlo_scenario_c(
            survey_df,
            rho=rho,
            replications=replications,
            base_seed=50000 + 1000 * method_index,
            wtp_method=method,
        )

        rows.append({
            "wtp_method": method,
            "mean_service_rate":
                mc["realised_service_rate"].mean(),
            "p2.5":
                mc["realised_service_rate"].quantile(0.025),
            "p97.5":
                mc["realised_service_rate"].quantile(0.975),
            "mean_total_vkt":
                mc["total_vkt"].mean(),
            "mean_final_fare_revenue":
                mc["final_fare_revenue"].mean(),
        })

    return pd.DataFrame(rows)


wtp_sensitivity = run_wtp_interpretation_sensitivity(
    survey,
    rho=rho,
    replications=100,
)

display(wtp_sensitivity)

## 15. Export paper-ready result tables

The files below are created in the Colab working directory and can be downloaded from the Colab Files panel.

In [ ]:
ABC_summary.to_csv(
    "ABC_summary_at_illustrative_rho.csv",
    index=False,
    encoding="utf-8-sig"
)

mc_C.to_csv(
    "scenario_C_monte_carlo_runs.csv",
    index=False,
    encoding="utf-8-sig"
)

discount_results.to_csv(
    "ABC_discount_sensitivity.csv",
    index=False,
    encoding="utf-8-sig"
)

wtp_sensitivity.to_csv(
    "scenario_C_WTP_sensitivity.csv",
    index=False,
    encoding="utf-8-sig"
)

request_level_C.to_csv(
    "scenario_C_single_run_request_level.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print("- ABC_summary_at_illustrative_rho.csv")
print("- scenario_C_monte_carlo_runs.csv")
print("- ABC_discount_sensitivity.csv")
print("- scenario_C_WTP_sensitivity.csv")
print("- scenario_C_single_run_request_level.csv")

## 16. Paper-ready result figures

The following figures visualise the most important observations from the A/B/C experiments.

They are intentionally focused on **results**, not decoration:

1. A/B/C realised service rate;
2. rejection decomposition;
3. Scenario C Monte Carlo distribution;
4. service-rate sensitivity to the common upfront discount;
5. VKT sensitivity to the common upfront discount.

Each figure is saved automatically at 300 dpi for later use in the paper.

In [ ]:
# Figure 1 — Realised service rate by behavioural scenario

c_mean_rate = mc_C["realised_service_rate"].mean()
c_low_rate = mc_C["realised_service_rate"].quantile(0.025)
c_high_rate = mc_C["realised_service_rate"].quantile(0.975)

scenario_names = [
    "A Universal",
    "B Homogeneous",
    "C Survey-informed",
]

service_rates = [
    result_A["summary"]["realised_service_rate"],
    result_B["summary"]["realised_service_rate"],
    c_mean_rate,
]

lower_errors = [
    0.0,
    0.0,
    c_mean_rate - c_low_rate,
]

upper_errors = [
    0.0,
    0.0,
    c_high_rate - c_mean_rate,
]

fig, ax = plt.subplots(figsize=(7.5, 5))

ax.bar(
    scenario_names,
    service_rates,
    yerr=[lower_errors, upper_errors],
    capsize=5,
)

ax.set_ylabel("Realised service rate")
ax.set_title(
    f"Realised service under alternative behavioural assumptions "
    f"(upfront discount = {rho:.0%})"
)
ax.set_ylim(0, 1)

for i, value in enumerate(service_rates):
    ax.text(
        i,
        value + 0.035,
        f"{value:.1%}",
        ha="center",
        va="bottom",
    )

fig.tight_layout()
fig.savefig(
    "Figure_1_ABC_realised_service_rate.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Figure 2 — Rejection decomposition

c_route_mean = mc_C[
    "route_infeasible_rejections"
].mean()

c_passenger_mean = mc_C[
    "passenger_unacceptable_rejections"
].mean()

route_rejections = [
    result_A["summary"]["route_infeasible_rejections"],
    result_B["summary"]["route_infeasible_rejections"],
    c_route_mean,
]

passenger_rejections = [
    result_A["summary"]["passenger_unacceptable_rejections"],
    result_B["summary"]["passenger_unacceptable_rejections"],
    c_passenger_mean,
]

accepted_services = [
    result_A["summary"]["realised_services"],
    result_B["summary"]["realised_services"],
    mc_C["realised_services"].mean(),
]

x = np.arange(
    len(scenario_names)
)

fig, ax = plt.subplots(figsize=(7.5, 5))

ax.bar(
    x,
    accepted_services,
    label="Realised service",
)

ax.bar(
    x,
    route_rejections,
    bottom=accepted_services,
    label="Route-infeasible rejection",
)

bottom_passenger = (
    np.array(accepted_services)
    + np.array(route_rejections)
)

ax.bar(
    x,
    passenger_rejections,
    bottom=bottom_passenger,
    label="Passenger-unacceptable rejection",
)

ax.set_xticks(
    x,
    scenario_names,
)

ax.set_ylabel("Number of requests")
ax.set_title("Request outcomes by behavioural scenario")
ax.set_ylim(0, len(requests_A))
ax.legend()

fig.tight_layout()
fig.savefig(
    "Figure_2_request_outcome_decomposition.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Figure 3 — Scenario C Monte Carlo distribution

fig, ax = plt.subplots(figsize=(7.5, 5))

ax.hist(
    mc_C["realised_service_rate"],
    bins=np.arange(
        -0.025,
        1.026,
        0.05,
    ),
    edgecolor="black",
)

ax.axvline(
    mc_C["realised_service_rate"].mean(),
    linestyle="--",
    linewidth=1.5,
    label=(
        "Mean = "
        f"{mc_C['realised_service_rate'].mean():.1%}"
    ),
)

ax.set_xlabel("Realised service rate")
ax.set_ylabel("Monte Carlo replications")
ax.set_title(
    "Distribution of realised service rate under "
    "survey-informed heterogeneity"
)
ax.set_xlim(0, 1)
ax.legend()

fig.tight_layout()
fig.savefig(
    "Figure_3_scenario_C_monte_carlo_distribution.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Figure 4 — Discount sensitivity of realised service

fig, ax = plt.subplots(figsize=(7.5, 5))

for scenario, group in discount_results.groupby(
    "scenario"
):
    group = group.sort_values(
        "rho"
    )

    ax.plot(
        group["rho"],
        group["realised_service_rate"],
        marker="o",
        label=scenario,
    )

c_group = discount_results[
    discount_results["scenario"]
    == "C Survey-informed"
].sort_values("rho")

ax.fill_between(
    c_group["rho"].to_numpy(),
    c_group["service_rate_low"].to_numpy(),
    c_group["service_rate_high"].to_numpy(),
    alpha=0.15,
)

ax.set_xlabel("Upfront pooled discount")
ax.set_ylabel("Realised service rate")
ax.set_title(
    "Sensitivity of realised service to upfront pooled discount"
)
ax.set_ylim(0, 1)
ax.legend()

fig.tight_layout()
fig.savefig(
    "Figure_4_discount_sensitivity_service_rate.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Figure 5 — Discount sensitivity of total VKT

fig, ax = plt.subplots(figsize=(7.5, 5))

for scenario, group in discount_results.groupby(
    "scenario"
):
    group = group.sort_values(
        "rho"
    )

    ax.plot(
        group["rho"],
        group["total_vkt"],
        marker="o",
        label=scenario,
    )

ax.set_xlabel("Upfront pooled discount")
ax.set_ylabel("Total vehicle kilometres travelled")
ax.set_title(
    "Operational VKT response to the common pricing environment"
)
ax.legend()

fig.tight_layout()
fig.savefig(
    "Figure_5_discount_sensitivity_total_vkt.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Optional: list all paper-ready figures created above

from pathlib import Path

figure_files = sorted(
    Path(".").glob("Figure_*.png")
)

print("Generated figures:")

for figure_file in figure_files:
    print("-", figure_file)

### Figure interpretation guide

These figures compare operational outputs across Scenarios A/B/C. Interpret them only after running the notebook with an authorised local dataset. No empirical result is bundled with this public notebook.


## 17. Reproducible large-scale synthetic case generator

The original 6-vehicle / 20-request experiment is retained as a small validation case.

For the formal simulation experiments, this notebook now adds a reproducible synthetic generator that creates:

- random initial vehicle locations;
- random passenger origins and destinations;
- random request arrival times;
- random party sizes;
- random vehicle capacities.

Every experiment uses a fixed random seed, so the same case can be reproduced exactly.

The default spatial pattern is a simple clustered urban-demand pattern. Origins and destinations are sampled around several synthetic activity centres, creating a mixture of shorter intra-area trips and longer inter-area trips.

The operational data are synthetic; Scenario C behaviour is generated from the locally supplied authorised survey sample.

For every case seed, Scenarios A/B/C receive exactly the same fleet, OD pairs, arrival times, capacities and pricing environment.

In [ ]:
def _clip_location(
    x,
    y,
    width,
    height
):
    return Location(
        float(
            np.clip(
                x,
                0.0,
                width
            )
        ),
        float(
            np.clip(
                y,
                0.0,
                height
            )
        ),
    )


def _sample_clustered_location(
    rng,
    centres,
    width,
    height,
    cluster_sd=4.5,
):
    # Sample one point around a synthetic activity centre.

    centre = centres[
        rng.integers(
            0,
            len(centres)
        )
    ]

    x = rng.normal(
        centre[0],
        cluster_sd
    )

    y = rng.normal(
        centre[1],
        cluster_sd
    )

    return _clip_location(
        x,
        y,
        width,
        height
    )


def _sample_uniform_location(
    rng,
    width,
    height,
):
    return Location(
        float(
            rng.uniform(
                0,
                width
            )
        ),
        float(
            rng.uniform(
                0,
                height
            )
        ),
    )


def build_synthetic_case(
    n_requests,
    n_vehicles,
    seed=2026,
    area_width=70.0,
    area_height=40.0,
    horizon=120.0,
    spatial_pattern="clustered",
    min_trip_distance=3.0,
):
    # Reproducible synthetic ride-pooling case.

    rng = np.random.default_rng(
        seed
    )

    centres = np.array(
        [
            [
                0.18 * area_width,
                0.22 * area_height
            ],
            [
                0.35 * area_width,
                0.72 * area_height
            ],
            [
                0.68 * area_width,
                0.30 * area_height
            ],
            [
                0.82 * area_width,
                0.78 * area_height
            ],
        ],
        dtype=float,
    )

    # -------------------------
    # Vehicles
    # -------------------------

    cars = []

    for car_id in range(
        n_vehicles
    ):

        if spatial_pattern == "clustered":

            if rng.random() < 0.5:

                location = (
                    _sample_clustered_location(
                        rng,
                        centres,
                        area_width,
                        area_height,
                        cluster_sd=6.0,
                    )
                )

            else:

                location = (
                    _sample_uniform_location(
                        rng,
                        area_width,
                        area_height,
                    )
                )

        elif spatial_pattern == "uniform":

            location = (
                _sample_uniform_location(
                    rng,
                    area_width,
                    area_height,
                )
            )

        else:

            raise ValueError(
                "spatial_pattern must be "
                "'clustered' or 'uniform'."
            )

        capacity = int(
            rng.choice(
                [
                    3,
                    4
                ],
                p=[
                    0.20,
                    0.80
                ]
            )
        )

        cars.append(
            Car(
                car_id=car_id,
                current_location=location,
                capacity=capacity,
            )
        )

    # -------------------------
    # Requests
    # -------------------------

    request_times = np.sort(
        rng.uniform(
            0.0,
            horizon,
            size=n_requests
        )
    )

    requests = []

    for request_id, request_time in enumerate(
        request_times
    ):

        if spatial_pattern == "clustered":

            origin = (
                _sample_clustered_location(
                    rng,
                    centres,
                    area_width,
                    area_height,
                    cluster_sd=4.5,
                )
            )

        else:

            origin = (
                _sample_uniform_location(
                    rng,
                    area_width,
                    area_height,
                )
            )

        destination = None

        for _ in range(
            200
        ):

            if spatial_pattern == "clustered":

                if rng.random() < 0.65:

                    destination = (
                        _sample_clustered_location(
                            rng,
                            centres,
                            area_width,
                            area_height,
                            cluster_sd=4.5,
                        )
                    )

                else:

                    destination = (
                        _clip_location(
                            rng.normal(
                                origin.x,
                                7.0
                            ),
                            rng.normal(
                                origin.y,
                                7.0
                            ),
                            area_width,
                            area_height,
                        )
                    )

            else:

                destination = (
                    _sample_uniform_location(
                        rng,
                        area_width,
                        area_height,
                    )
                )

            if (
                physical_distance(
                    origin,
                    destination
                )
                >= min_trip_distance
            ):
                break

        party_size = int(
            rng.choice(
                [
                    1,
                    2
                ],
                p=[
                    0.80,
                    0.20
                ]
            )
        )

        requests.append(
            Request(
                request_id=request_id,
                t_r=float(
                    request_time
                ),
                o_r=origin,
                d_r=destination,
                party_size=party_size,
            )
        )

    return (
        cars,
        requests
    )


# Reproducibility test
cars_syn_1, req_syn_1 = (
    build_synthetic_case(
        n_requests=50,
        n_vehicles=15,
        seed=2026,
    )
)

cars_syn_2, req_syn_2 = (
    build_synthetic_case(
        n_requests=50,
        n_vehicles=15,
        seed=2026,
    )
)

assert (
    (cars_syn_1[0].current_location.x, cars_syn_1[0].current_location.y)
    ==
    (cars_syn_2[0].current_location.x, cars_syn_2[0].current_location.y)
)

assert (
    (req_syn_1[0].o_r.x, req_syn_1[0].o_r.y)
    ==
    (req_syn_2[0].o_r.x, req_syn_2[0].o_r.y)
)

print(
    "Synthetic generator reproducibility check passed."
)

### Visual check of one generated synthetic case

This figure shows the generated vehicle locations and passenger OD points. It is mainly a generator-validation figure rather than a core Results figure.

In [ ]:
cars_demo, requests_demo = (
    build_synthetic_case(
        n_requests=100,
        n_vehicles=30,
        seed=2026,
    )
)

fig, ax = plt.subplots(
    figsize=(
        8,
        5
    )
)

ax.scatter(
    [
        c.current_location.x
        for c in cars_demo
    ],
    [
        c.current_location.y
        for c in cars_demo
    ],
    marker="s",
    label="Initial vehicle location",
)

ax.scatter(
    [
        r.o_r.x
        for r in requests_demo
    ],
    [
        r.o_r.y
        for r in requests_demo
    ],
    marker="o",
    alpha=0.55,
    label="Request origin",
)

ax.scatter(
    [
        r.d_r.x
        for r in requests_demo
    ],
    [
        r.d_r.y
        for r in requests_demo
    ],
    marker="x",
    alpha=0.55,
    label="Request destination",
)

ax.set_xlabel(
    "Synthetic x-coordinate"
)

ax.set_ylabel(
    "Synthetic y-coordinate"
)

ax.set_title(
    "Example synthetic fleet and passenger OD distribution"
)

ax.legend()

fig.tight_layout()

fig.savefig(
    "Figure_6_synthetic_case_spatial_distribution.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 18. Large-scale experimental matrix

The default matrix uses:

| Case | Requests | Vehicles |
|---|---:|---:|
| Low | 50 | 15 |
| Medium | 100 | 30 |
| High | 200 | 60 |

This keeps the request-to-fleet ratio approximately constant while system scale increases.

In [ ]:
SYNTHETIC_CASES = {
    "Low": {
        "n_requests": 50,
        "n_vehicles": 15,
    },

    "Medium": {
        "n_requests": 100,
        "n_vehicles": 30,
    },

    "High": {
        "n_requests": 200,
        "n_vehicles": 60,
    },
}

SYNTHETIC_SEEDS = [
    4101,
    4102,
    4103,
    4104,
    4105,
]

display(
    pd.DataFrame(
        SYNTHETIC_CASES
    ).T
)

## 19. Run A/B/C on the same synthetic case

For a given synthetic case seed, A/B/C are rebuilt using that exact seed. Therefore all operational inputs are identical and only behavioural parameterisation changes.

In [ ]:
def run_abc_on_synthetic_case(
    n_requests,
    n_vehicles,
    case_seed,
    rho,
    profiles,
    profile_seed,
    spatial_pattern="clustered",
):
    outputs = []

    # -------------------------
    # Scenario A
    # -------------------------

    cars_A, requests_A = (
        build_synthetic_case(
            n_requests=n_requests,
            n_vehicles=n_vehicles,
            seed=case_seed,
            spatial_pattern=spatial_pattern,
        )
    )

    result_A = run_simulation(
        cars_A,
        requests_A,
        mode="universal",
        rho=rho,
        hom=homogeneous_params,
    )

    # -------------------------
    # Scenario B
    # -------------------------

    cars_B, requests_B = (
        build_synthetic_case(
            n_requests=n_requests,
            n_vehicles=n_vehicles,
            seed=case_seed,
            spatial_pattern=spatial_pattern,
        )
    )

    result_B = run_simulation(
        cars_B,
        requests_B,
        mode="homogeneous",
        rho=rho,
        hom=homogeneous_params,
    )

    # -------------------------
    # Scenario C
    # -------------------------

    cars_C, requests_C = (
        build_synthetic_case(
            n_requests=n_requests,
            n_vehicles=n_vehicles,
            seed=case_seed,
            spatial_pattern=spatial_pattern,
        )
    )

    requests_C = (
        assign_complete_profiles(
            requests_C,
            profiles,
            seed=profile_seed,
        )
    )

    result_C = run_simulation(
        cars_C,
        requests_C,
        mode="survey_heterogeneous",
        rho=rho,
        hom=homogeneous_params,
    )

    for scenario_name, result in [
        (
            "A Universal",
            result_A
        ),
        (
            "B Homogeneous",
            result_B
        ),
        (
            "C Survey-informed",
            result_C
        ),
    ]:

        row = dict(
            result[
                "summary"
            ]
        )

        row[
            "scenario"
        ] = scenario_name

        row[
            "case_seed"
        ] = case_seed

        row[
            "profile_seed"
        ] = (
            profile_seed
            if scenario_name
            == "C Survey-informed"
            else np.nan
        )

        outputs.append(
            row
        )

    return pd.DataFrame(
        outputs
    )

## 20. Quick large-scale run

This runs one reproducible seed for each case size to verify the large-scale pipeline before launching repeated experiments.

In [ ]:
quick_large_scale_results = []

for case_name, settings in (
    SYNTHETIC_CASES.items()
):

    case_result = (
        run_abc_on_synthetic_case(
            n_requests=settings[
                "n_requests"
            ],
            n_vehicles=settings[
                "n_vehicles"
            ],
            case_seed=4101,
            rho=rho,
            profiles=profiles_main,
            profile_seed=5101,
        )
    )

    case_result[
        "case"
    ] = case_name

    quick_large_scale_results.append(
        case_result
    )

quick_large_scale_results = (
    pd.concat(
        quick_large_scale_results,
        ignore_index=True,
    )
)

display(
    quick_large_scale_results[
        [
            "case",
            "scenario",
            "requests",
            "nominal_matching_rate",
            "realised_service_rate",
            "route_infeasible_rejections",
            "passenger_unacceptable_rejections",
            "total_vkt",
            "final_fare_revenue",
            "distance_weighted_occupancy",
        ]
    ]
)

## 21. Repeated synthetic experiments

A single random OD sample is not sufficient evidence.

The function below repeats the experiment over multiple synthetic case seeds. Scenario C is also repeated over multiple survey-profile assignments.

Recommended use:

- quick development: 5 case seeds × 5 Scenario-C profile replications;
- final paper: increase both replication counts if runtime permits.

In [ ]:
def run_repeated_synthetic_experiments(
    case_matrix=SYNTHETIC_CASES,
    case_seeds=SYNTHETIC_SEEDS,
    c_profile_replications=5,
    rho_value=None,
    wtp_method=MAIN_WTP_METHOD,
    base_profile_seed=80000,
    spatial_pattern="clustered",
):
    if rho_value is None:

        rho_value = rho

    rows = []

    for case_name, settings in (
        case_matrix.items()
    ):

        for case_seed in case_seeds:

            # -------------------------
            # Scenario A
            # -------------------------

            cars_A, requests_A = (
                build_synthetic_case(
                    n_requests=settings[
                        "n_requests"
                    ],
                    n_vehicles=settings[
                        "n_vehicles"
                    ],
                    seed=case_seed,
                    spatial_pattern=(
                        spatial_pattern
                    ),
                )
            )

            result_A = run_simulation(
                cars_A,
                requests_A,
                mode="universal",
                rho=rho_value,
                hom=homogeneous_params,
            )

            row_A = dict(
                result_A[
                    "summary"
                ]
            )

            row_A.update(
                {
                    "case":
                        case_name,

                    "scenario":
                        "A Universal",

                    "case_seed":
                        case_seed,

                    "profile_replication":
                        np.nan,

                    "profile_seed":
                        np.nan,
                }
            )

            rows.append(
                row_A
            )

            # -------------------------
            # Scenario B
            # -------------------------

            cars_B, requests_B = (
                build_synthetic_case(
                    n_requests=settings[
                        "n_requests"
                    ],
                    n_vehicles=settings[
                        "n_vehicles"
                    ],
                    seed=case_seed,
                    spatial_pattern=(
                        spatial_pattern
                    ),
                )
            )

            result_B = run_simulation(
                cars_B,
                requests_B,
                mode="homogeneous",
                rho=rho_value,
                hom=homogeneous_params,
            )

            row_B = dict(
                result_B[
                    "summary"
                ]
            )

            row_B.update(
                {
                    "case":
                        case_name,

                    "scenario":
                        "B Homogeneous",

                    "case_seed":
                        case_seed,

                    "profile_replication":
                        np.nan,

                    "profile_seed":
                        np.nan,
                }
            )

            rows.append(
                row_B
            )

            # -------------------------
            # Scenario C
            # -------------------------

            for rep in range(
                c_profile_replications
            ):

                profile_seed = (
                    base_profile_seed
                    + case_seed * 100
                    + rep
                )

                profiles_C_rep = (
                    build_survey_profiles(
                        survey,
                        wtp_method=(
                            wtp_method
                        ),
                        seed=profile_seed,
                    )
                )

                cars_C, requests_C = (
                    build_synthetic_case(
                        n_requests=settings[
                            "n_requests"
                        ],
                        n_vehicles=settings[
                            "n_vehicles"
                        ],
                        seed=case_seed,
                        spatial_pattern=(
                            spatial_pattern
                        ),
                    )
                )

                requests_C = (
                    assign_complete_profiles(
                        requests_C,
                        profiles_C_rep,
                        seed=(
                            profile_seed
                            + 1_000_000
                        ),
                    )
                )

                result_C = run_simulation(
                    cars_C,
                    requests_C,
                    mode=(
                        "survey_heterogeneous"
                    ),
                    rho=rho_value,
                    hom=homogeneous_params,
                )

                row_C = dict(
                    result_C[
                        "summary"
                    ]
                )

                row_C.update(
                    {
                        "case":
                            case_name,

                        "scenario":
                            "C Survey-informed",

                        "case_seed":
                            case_seed,

                        "profile_replication":
                            rep,

                        "profile_seed":
                            profile_seed,
                    }
                )

                rows.append(
                    row_C
                )

    return pd.DataFrame(
        rows
    )


# QUICK RUN:
# 5 synthetic case seeds × 5 Scenario-C profile replications

synthetic_repeated = (
    run_repeated_synthetic_experiments(
        case_seeds=(
            SYNTHETIC_SEEDS
        ),
        c_profile_replications=5,
    )
)

print(
    "Completed rows:",
    len(
        synthetic_repeated
    )
)

## 22. Summarise repeated large-scale experiments

In [ ]:
synthetic_summary = (
    synthetic_repeated
    .groupby(
        [
            "case",
            "scenario"
        ],
        as_index=False
    )
    .agg(
        mean_nominal_matching_rate=(
            "nominal_matching_rate",
            "mean"
        ),

        mean_realised_service_rate=(
            "realised_service_rate",
            "mean"
        ),

        sd_realised_service_rate=(
            "realised_service_rate",
            "std"
        ),

        mean_total_vkt=(
            "total_vkt",
            "mean"
        ),

        mean_final_fare_revenue=(
            "final_fare_revenue",
            "mean"
        ),

        mean_occupied_vkt_share=(
            "occupied_vkt_share",
            "mean"
        ),

        mean_distance_weighted_occupancy=(
            "distance_weighted_occupancy",
            "mean"
        ),

        mean_route_rejections=(
            "route_infeasible_rejections",
            "mean"
        ),

        mean_passenger_rejections=(
            "passenger_unacceptable_rejections",
            "mean"
        ),
    )
)

display(
    synthetic_summary
)

## 23. Figure — A/B/C realised service across synthetic system scales

In [ ]:
case_order = [
    "Low",
    "Medium",
    "High"
]

scenario_order = [
    "A Universal",
    "B Homogeneous",
    "C Survey-informed"
]

x = np.arange(
    len(
        case_order
    )
)

bar_width = 0.24

fig, ax = plt.subplots(
    figsize=(
        9,
        5.5
    )
)

for scenario_index, scenario in enumerate(
    scenario_order
):

    scenario_rows = (
        synthetic_summary[
            synthetic_summary[
                "scenario"
            ]
            == scenario
        ]
        .set_index(
            "case"
        )
        .reindex(
            case_order
        )
    )

    offsets = (
        x
        + (
            scenario_index - 1
        )
        * bar_width
    )

    values = scenario_rows[
        "mean_realised_service_rate"
    ].to_numpy()

    errors = scenario_rows[
        "sd_realised_service_rate"
    ].fillna(
        0.0
    ).to_numpy()

    ax.bar(
        offsets,
        values,
        width=bar_width,
        yerr=errors,
        capsize=3,
        label=scenario,
    )

ax.set_xticks(
    x,
    [
        "Low\n50 requests / 15 vehicles",
        "Medium\n100 / 30",
        "High\n200 / 60",
    ],
)

ax.set_ylabel(
    "Mean realised service rate"
)

ax.set_ylim(
    0,
    1
)

ax.set_title(
    "A/B/C behavioural assumptions across synthetic system scales"
)

ax.legend()

fig.tight_layout()

fig.savefig(
    "Figure_7_large_scale_ABC_service_rate.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 24. Figure — passenger-unacceptable rejection across synthetic scales

This directly visualises the behavioural rejection mechanism introduced by the new model.

In [ ]:
plot_df = (
    synthetic_repeated.copy()
)

plot_df[
    "passenger_unacceptable_rate"
] = (
    plot_df[
        "passenger_unacceptable_rejections"
    ]
    / plot_df[
        "requests"
    ]
)

behaviour_rejection_summary = (
    plot_df
    .groupby(
        [
            "case",
            "scenario"
        ],
        as_index=False
    )
    .agg(
        mean_passenger_unacceptable_rate=(
            "passenger_unacceptable_rate",
            "mean"
        )
    )
)

fig, ax = plt.subplots(
    figsize=(
        8.5,
        5
    )
)

for scenario in scenario_order:

    group = (
        behaviour_rejection_summary[
            behaviour_rejection_summary[
                "scenario"
            ]
            == scenario
        ]
        .set_index(
            "case"
        )
        .reindex(
            case_order
        )
    )

    ax.plot(
        case_order,
        group[
            "mean_passenger_unacceptable_rate"
        ],
        marker="o",
        label=scenario,
    )

ax.set_xlabel(
    "Synthetic case scale"
)

ax.set_ylabel(
    "Passenger-unacceptable rejection rate"
)

ax.set_ylim(
    0,
    1
)

ax.set_title(
    "Behavioural rejection across synthetic system scales"
)

ax.legend()

fig.tight_layout()

fig.savefig(
    "Figure_8_large_scale_passenger_rejection.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 25. Optional full large-scale discount sensitivity

This crosses the synthetic case matrix with multiple common upfront discounts. It is intentionally not run automatically because it can take considerably longer.

In [ ]:
def run_large_scale_discount_sensitivity(
    rho_grid=(
        0.45,
        0.55,
        0.65,
        0.75,
        0.85
    ),
    case_seeds=(
        4101,
        4102,
        4103
    ),
    c_profile_replications=3,
):
    outputs = []

    for rho_value in rho_grid:

        result = (
            run_repeated_synthetic_experiments(
                case_seeds=list(
                    case_seeds
                ),
                c_profile_replications=(
                    c_profile_replications
                ),
                rho_value=rho_value,
                base_profile_seed=(
                    120000
                    + int(
                        rho_value
                        * 10000
                    )
                ),
            )
        )

        result[
            "pricing_discount"
        ] = rho_value

        outputs.append(
            result
        )

    return pd.concat(
        outputs,
        ignore_index=True
    )


# Example:
#
# large_discount_results = (
#     run_large_scale_discount_sensitivity()
# )
#
# display(
#     large_discount_results.head()
# )

## 26. Export large-scale synthetic results

In [ ]:
synthetic_repeated.to_csv(
    "synthetic_repeated_ABC_results.csv",
    index=False,
    encoding="utf-8-sig"
)

synthetic_summary.to_csv(
    "synthetic_ABC_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Saved large-scale synthetic result tables."
)

### How to describe this experiment in the paper

These experiments should be described as **controlled synthetic simulation experiments**.

The random OD and fleet generator is not intended to represent an observed city. Its purpose is to test whether the behavioural conclusion remains robust when:

- system scale increases;
- vehicle initial positions change;
- passenger OD patterns change;
- request arrival times change;
- respondent-profile assignments change.

Any empirical component must come from an authorised survey supplied locally by the user.

## 27. Interpretation cautions before final paper Results

1. The current 6-vehicle / 20-request case is only an initial validation case.
2. The current spatial network is synthetic Euclidean space rather than a real road network.
3. When separate \(W_i^{max}\) and \(\Delta T_i^{max}\) are not supplied, the undergraduate `sigma=0.5` combined service-time rule is retained.
4. The current upfront pricing environment is intentionally separated from passenger acceptance. Do not change pricing between A/B/C.
5. Scenario B is a hard-threshold homogeneous benchmark. Under a constant \(\rho\), it can show a sharp threshold response; this is part of what the heterogeneous Scenario C is intended to test.
6. The ordinal-logit model is **not multiplied into** the same Q16/Q34 hard acceptance rule. It remains a behavioural-analysis / robustness component and is not double-counted here.
7. Before the final paper experiment, expand demand and fleet conditions and pre-specify all pricing / WTP sensitivity settings.